# LearnMate — M0 environment smoke test
Runs on a free Colab **T4** (Runtime → Change runtime type → GPU). Checkpoints and caches are written to Google Drive so a disconnect never loses work.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content
!git clone -b claude/wonderful-bohr-1idsaa https://github.com/KirtirajChaudhari/LearnMate.git || (cd LearnMate && git pull)
%cd /content/LearnMate

/content
Cloning into 'LearnMate'...
remote: Enumerating objects: 196, done.
remote: Counting objects: 100% (196/196), done.
remote: Compressing objects: 100% (133/133), done.
remote: Total 196 (delta 60), reused 186 (delta 55), pack-reused 0 (from 0)
Receiving objects: 100% (196/196), 1.44 MiB | 4.47 MiB/s, done.
Resolving deltas: 100% (60/60), done.
/content/LearnMate


In [3]:
!pip -q install -e '.[gpu,eval,dev]'
!nvidia-smi --query-gpu=name,memory.total --format=csv

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for learnmate (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [4]:
# persist large/expensive artifacts on Drive
import os
D='/content/drive/MyDrive/learnmate'
for sub in ['data_raw','data_processed','data_cache','models','results']:
    os.makedirs(f'$D/{sub}', exist_ok=True)
!mkdir -p data && ln -sfn $D/data_raw data/raw && ln -sfn $D/data_processed data/processed && ln -sfn $D/data_cache data/cache && ln -sfn $D/models models && ln -sfn $D/results results

In [5]:
!test -d data/raw/d2l-en || git clone https://github.com/d2l-ai/d2l-en.git data/raw/d2l-en
!git -C data/raw/d2l-en checkout -q 23d7a5aecceee57d1292c56e90cce307f183bb0a
!python -m learnmate.ingest --check
!python scripts/build_squad.py

fatal: could not create leading directories of 'data/raw/d2l-en': No such file or directory
fatal: cannot change to 'data/raw/d2l-en': No such file or directory
fatal: cannot change to '/content/LearnMate/data/raw/d2l-en': No such file or directory
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/LearnMate/learnmate/ingest/__main__.py", line 73, in <module>
    main()
    ~~~~^^
  File "/content/LearnMate/learnmate/ingest/__main__.py", line 31, in main
    head = git_head(repo)
  File "/content/LearnMate/learnmate/ingest/__main__.py", line 20, in git_head
    return subprocess.check_output(["git", "-C", str(path), "rev-parse", "HEAD"], text=True).strip()
           ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/subprocess.py", line 473, in check_output
    return run(*popenargs, stdout=PIPE, timeout=timeout, check=Tru

Loads every model once, runs a forward pass and records latency and **peak VRAM** to `results/m0_smoke.json`. The fp16 vs NF4 result decides the frozen generator runtime (ADR-003).

In [6]:
!python scripts/smoke_models.py
!cat results/m0_smoke.json

I0000 00:00:1795458136.813533    1532 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795458140.758546    1532 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
modules.json: 100% 349/349 [00:00<00:00, 1.57MB/s]
config_sentence_transformers.json: 100% 124/124 [00:00<00:00, 826kB/s]
README.md: 94.6kB [00:00, 80.7MB/s]
sentence_bert_config.json: 100% 52.0/52.0 [00:00<00:00, 278kB/s]
config.json: 100% 777/777 [00:00<00:00, 4.73MB/s]
model.safetensors: 100% 438M/438M [00:04<00:00, 99.7MB/s]
tokenizer_config.json: 100% 366/366 [00:00<00:00, 2.04MB/s]
vocab.txt: 232kB [00:00, 23.1MB/s]
tokenizer.json: 711kB [00:00, 63.1MB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 462kB/s]
config.json: 100% 190/190 [00:00<00:00, 776kB/

In [7]:
!pip freeze > requirements.lock && head -50 requirements.lock

absl-py==2.0.0
accelerate==1.15.0
access==1.1.10.post3
affine==3.0.1
aiofiles==25.1.0
aiohappyeyeballs==2.7.1
aiohttp==3.14.5
aiosignal==1.4.0
aiosqlite==0.22.1
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.12.1
altair==5.5.0
annotated-doc==0.0.5
annotated-types==0.8.0
anyio==4.16.0
anywidget==0.9.21
apsw==3.54.1.0
argon2-cffi==25.1.0
argon2-cffi-bindings==26.1.0
array_record==0.8.6
arrow==1.4.0
arviz==0.22.0
astropy==7.2.3
astropy-iers-data==0.2026.11.2.0.34.55
astunparse==1.6.3
atpublic==5.1
attrs==26.1.0
audioop-lts==0.2.2
audioread==3.1.0
Authlib==1.9.3
autograd==1.9.1
babel==2.18.0
backcall==0.2.0
beartype==0.23.1
beautifulsoup4==4.13.5
betterproto==2.0.0b7
bigframes==2.51.0
bigquery-magics==0.14.0
bitsandbytes==0.50.2
bleach==6.4.0
blinker==1.9.0
blis==1.3.4
blobfile==3.3.0
blosc2==4.14.1
bokeh==3.8.2
Bottleneck==1.4.2
bqplot==0.12.47
branca==0.8.2


In [8]:
!python -m pytest -q

...........................s...                                          [100%]
30 passed, 2 skipped in 3.67s
